In [8]:
import numpy as np
import torch
from scipy.optimize import linear_sum_assignment as scipy_linear_sum_assignment
from geomloss import SamplesLoss
import time

def solve_assignment_geomloss(cost_matrix_np, blur_value=0.01, verbose=True, run_scipy_reference=True):
    """
    Solves the linear sum assignment problem approximately using GeomLoss,
    extracting the hard assignment via a GPU-based greedy method from the soft plan.

    Args:
        cost_matrix_np (np.ndarray): The N x M cost matrix.
        blur_value (float): Regularization strength for Sinkhorn algorithm.
        verbose (bool): Whether to print intermediate results.
        run_scipy_reference (bool): Whether to compute and print the exact SciPy solution.

    Returns:
        tuple: (row_indices, col_indices, total_cost) for the approximate assignment.
    """
    N, M = cost_matrix_np.shape
    if verbose:
        print(f"Input cost matrix (N={N}, M={M}) shape.")

    # --- 1. Exact solution with SciPy (for reference) ---
    if run_scipy_reference:
        t_scipy_start = time.time()
        row_ind_ref, col_ind_ref = scipy_linear_sum_assignment(cost_matrix_np)
        cost_ref = cost_matrix_np[row_ind_ref, col_ind_ref].sum()
        t_scipy_end = time.time()
        if verbose:
            print("--- SciPy Exact Solution ---")
            if N * M < 100:
                print(f"Row indices: {row_ind_ref}")
                print(f"Col indices: {col_ind_ref}")
            else:
                print(f"Row/Col indices: (suppressed for large matrix)")
            print(f"Total cost: {cost_ref}")
            print(f"SciPy execution time: {t_scipy_end - t_scipy_start:.4f}s\n")
    elif verbose:
        print("--- SciPy Exact Solution: Skipped ---\n")

    # --- 2. Prepare inputs for GeomLoss ---
    t_geomloss_prep_start = time.time()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if verbose:
        print(f"Using device: {device}")
    cost_matrix_torch = torch.from_numpy(cost_matrix_np).float().to(device)

    alpha = torch.ones(N, dtype=torch.float32, device=device) / N
    beta = torch.ones(M, dtype=torch.float32, device=device) / M
    
    x_coords = torch.arange(N, dtype=torch.float32, device=device).view(N, 1)
    y_coords = torch.arange(M, dtype=torch.float32, device=device).view(M, 1)

    cost_function = lambda x_bnd, y_bmd: cost_matrix_torch.unsqueeze(0)
    t_geomloss_prep_end = time.time()

    # --- 3. Initialize and run SamplesLoss (Sinkhorn) ---
    t_sinkhorn_start = time.time()
    loss_fn = SamplesLoss(
        loss="sinkhorn",
        p=1,
        blur=blur_value,
        cost=cost_function,
        potentials=True,
        debias=False,
        backend="tensorized"
    )

    try:
        f_pot, g_pot = loss_fn(alpha, x_coords, beta, y_coords)
    except Exception as e:
        print(f"Error during SamplesLoss computation: {e}")
        return None, None, float('inf')
    t_sinkhorn_end = time.time()

    # --- 4. Reconstruct the transport plan P (on GPU) ---
    t_plan_recon_start = time.time()
    C_batch = cost_matrix_torch.unsqueeze(0) 
    # transport_plan_batched will be on the same device as f_pot, g_pot, C_batch (i.e., GPU if available)
    transport_plan_batched = torch.exp(
        (f_pot.unsqueeze(2) + g_pot.unsqueeze(1) - C_batch) / blur_value
    )
    # No transport_plan_np yet, keep on GPU for greedy extraction
    t_plan_recon_end = time.time()

    if verbose:
        print(f"--- GeomLoss Approximate Solution (blur={blur_value}) ---")

    # --- 5. GPU-based Greedy Assignment Extraction ---
    # Operates on transport_plan_batched[0] directly on GPU.
    # Complexity: Loop min(N,M) times. Inside: argmax (NM on GPU), masking.
    # Potentially faster than CPU sort O(NM log NM) for large NM.
    t_greedy_gpu_start = time.time()
    
    # Assuming batch size B=1 for the plan
    gpu_plan_full_precision = transport_plan_batched[0].clone() # Work with a clone

    num_assignments_to_make = min(N, M)
    
    row_ind_geomloss_list = [] # Collect indices on CPU
    col_ind_geomloss_list = []
    assignments_count = 0

    # For masking, we modify a working copy of the plan
    # Using float32 for the working plan should be fine.
    working_plan_gpu = gpu_plan_full_precision.float()


    for i in range(num_assignments_to_make):
        if assignments_count == num_assignments_to_make:
            break
        
        # Find the max value in the current working_plan_gpu
        # torch.argmax on a 2D tensor returns the flat index of the maximum value.
        flat_idx = torch.argmax(working_plan_gpu)
        
        # Convert flat index to 2D indices (row, column)
        r_best, c_best = torch.unravel_index(flat_idx, working_plan_gpu.shape)
        
        # Get the actual max value to check if it's valid (not -inf)
        current_max_val = working_plan_gpu[r_best, c_best]

        if current_max_val == -torch.inf or current_max_val == -float('inf'): # Check if all valid options are exhausted
            if verbose:
                print(f"GPU Greedy: No more assignable elements (max is -inf) at iteration {assignments_count + 1}/{num_assignments_to_make}.")
            break
            
        row_ind_geomloss_list.append(r_best.item()) # .item() moves scalar tensor to CPU Python number
        col_ind_geomloss_list.append(c_best.item())
        assignments_count += 1
        
        # Mask the selected row and column in working_plan_gpu for the next iteration
        working_plan_gpu[r_best, :] = -torch.inf 
        working_plan_gpu[:, c_best] = -torch.inf

    row_ind_geomloss = np.array(row_ind_geomloss_list)
    col_ind_geomloss = np.array(col_ind_geomloss_list)
    t_greedy_gpu_end = time.time()
    
    cost_geomloss = float('inf')
    if len(row_ind_geomloss) > 0 :
        if len(row_ind_geomloss) != num_assignments_to_make and verbose:
            print(f"Warning: GPU Greedy assignment found {len(row_ind_geomloss)} pairs, expected {num_assignments_to_make}.")
        # Cost calculation uses original cost_matrix_np (CPU)
        cost_geomloss = cost_matrix_np[row_ind_geomloss, col_ind_geomloss].sum()
    elif verbose:
        print("Warning: GPU Greedy assignment found 0 pairs.")

    if verbose:
        if N * M < 100:
            print(f"Row indices (GPU greedy): {row_ind_geomloss}")
            print(f"Col indices (GPU greedy): {col_ind_geomloss}")
        else:
            print(f"Row/Col indices (GPU greedy): (suppressed for large matrix)")
        print(f"Total cost (GPU greedy): {cost_geomloss}")
        print(f"GeomLoss Prep time: {t_geomloss_prep_end - t_geomloss_prep_start:.4f}s")
        print(f"Sinkhorn execution time: {t_sinkhorn_end - t_sinkhorn_start:.4f}s")
        print(f"Plan reconstruction time (GPU): {t_plan_recon_end - t_plan_recon_start:.4f}s")
        print(f"GPU Greedy assignment extraction time: {t_greedy_gpu_end - t_greedy_gpu_start:.4f}s")
        total_geomloss_time = (t_geomloss_prep_end - t_geomloss_prep_start) + \
                              (t_sinkhorn_end - t_sinkhorn_start) + \
                              (t_plan_recon_end - t_plan_recon_start) + \
                              (t_greedy_gpu_end - t_greedy_gpu_start)
        print(f"Total GeomLoss + GPU Greedy time: {total_geomloss_time:.4f}s\n")

    return row_ind_geomloss, col_ind_geomloss, cost_geomloss

def run_large_scale_test(N_large=500, M_large=500, run_scipy_for_large_test=False, blur_val=0.01):
    """
    Runs a test case with a larger, randomly generated cost matrix.
    GPU-based greedy should be faster than CPU sort for large N,M.
    Sinkhorn iterations or memory for cost_matrix_torch could still be limiting for very large N,M.
    """
    print(f"\n========= Test Case: Large Scale ({N_large}x{M_large}) =========")
    
    if run_scipy_for_large_test and N_large**3 > 1e8: # O(N^3) for SciPy
         print("Warning: Running SciPy exact solution for large matrix, this can be very slow (O(N^3)).")

    print(f"SciPy reference will be {'run' if run_scipy_for_large_test else 'skipped'}.")
    
    np.random.seed(42) 
    C_large = np.random.rand(N_large, M_large).astype(np.float32) * 100 
    
    solve_assignment_geomloss(C_large, blur_value=blur_val, verbose=True, run_scipy_reference=run_scipy_for_large_test)


if __name__ == '__main__':
    # --- Test Case 1: Square Matrix ---
    print("========= Test Case 1: Square Matrix =========")
    C1 = np.array([
        [9, 2, 7, 8],
        [6, 4, 3, 7],
        [5, 8, 1, 8],
        [7, 6, 9, 4]
    ], dtype=float)
    solve_assignment_geomloss(C1, blur_value=0.01)

    # --- Test Case 2: Rectangular Matrix (N < M) ---
    # ... (other small test cases remain the same)
    print("========= Test Case 2: Rectangular Matrix (N < M) =========")
    C2 = np.array([
        [10, 2, 8, 4],
        [1, 9, 7, 3],
        [5, 6, 12, 10]
    ], dtype=float)
    solve_assignment_geomloss(C2, blur_value=0.01)
    
    print("========= Test Case 3: Rectangular Matrix (N > M) =========")
    C3 = np.array([
        [10, 2, 8],
        [1, 9, 7],
        [5, 6, 12],
        [3, 4, 5]
    ], dtype=float)
    solve_assignment_geomloss(C3, blur_value=0.01)
    
    print("========= Test Case 4: Simple Known Optimal =========")
    C4 = np.array([
        [4, 1, 3], 
        [2, 0, 5], 
        [3, 2, 2]
    ], dtype=float)
    solve_assignment_geomloss(C4, blur_value=0.01)

    # --- Test Case 5: Large Scale ---
    # Test with N=500. For N=5000, memory for C_large (5000x5000 float32 = 100MB)
    # Sinkhorn on GPU might handle 5kx5k.
    # GPU greedy: 5000 iterations. Inside: argmax on 5000x5000 plan.
    run_large_scale_test(N_large=200, M_large=200, run_scipy_for_large_test=False, blur_val=0.01) 
    # run_large_scale_test(N_large=500, M_large=500, run_scipy_for_large_test=False, blur_val=0.01)
    
    # For the 5000x5000 test case from your output:
    print("\nRe-running user's 5000x5000 test case parameters:")
    run_large_scale_test(N_large=5000, M_large=5000, run_scipy_for_large_test=True, blur_val=0.01)
    # Expectation: GPU greedy time should be lower than the previous 6.98s CPU sort-greedy.
    # SciPy time might vary.


========= Test Case 1: Square Matrix =========
Input cost matrix (N=4, M=4) shape.
--- SciPy Exact Solution ---
Row indices: [0 1 2 3]
Col indices: [1 0 2 3]
Total cost: 13.0
SciPy execution time: 0.0001s

Using device: cuda
--- GeomLoss Approximate Solution (blur=0.01) ---
Row indices (GPU greedy): [0 3 2 1]
Col indices (GPU greedy): [1 3 2 0]
Total cost (GPU greedy): 13.0
GeomLoss Prep time: 0.0004s
Sinkhorn execution time: 0.0041s
Plan reconstruction time (GPU): 0.0000s
GPU Greedy assignment extraction time: 0.2705s
Total GeomLoss + GPU Greedy time: 0.2751s

========= Test Case 2: Rectangular Matrix (N < M) =========
Input cost matrix (N=3, M=4) shape.
--- SciPy Exact Solution ---
Row indices: [0 1 2]
Col indices: [1 3 0]
Total cost: 10.0
SciPy execution time: 0.0000s

Using device: cuda
Error during SamplesLoss computation: The size of tensor a (3) must match the size of tensor b (4) at non-singleton dimension 2
========= Test Case 3: Rectangular Matrix (N > M) =========
Input cost